# Priorizar hipótesis y evaluar una variante

**Brenis Hernández · Proyecto de portafolio**

Esta copia pública conserva el código y las explicaciones. Se retiraron las salidas originales para evitar publicar registros o identificadores; los resultados resumidos se documentan en la ficha del portafolio a partir de la versión original revisada. No se ha vuelto a ejecutar sobre los datos originales. También se omiten mensajes de revisión académica.

**Alcance:** La métrica del notebook es pedidos por visita, no conversión de usuarios únicos. Las visitas están agregadas; excluir usuarios cruzados de los pedidos no permite ajustar sus visitas de forma equivalente. Mann–Whitney compara distribuciones/rangos, no prueba directamente igualdad de medias. Los porcentajes son diferencias observadas del experimento, no una mejora implementada por mí.


# SPRINT 11 - PROYECTO

## Introduccion

Eres analista en una gran tienda online. Junto con el departamento de marketing has recopilado una lista de hipótesis que pueden ayudar a aumentar los ingresos.

Tienes que priorizar estas hipótesis, lanzar un test A/B y analizar los resultados.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

hyp = pd.read_csv('/datasets/hypotheses_us.csv', sep=';')  # importante
orders = pd.read_csv('/datasets/orders_us.csv')
visits = pd.read_csv('/datasets/visits_us.csv')

orders['date'] = pd.to_datetime(orders['date'])
visits['date'] = pd.to_datetime(visits['date'])

hyp.head(), orders.head(), visits.head()



Se cargaron los datasets. El archivo hypotheses_us.csv está separado por ;, por lo que se especificó sep=';'. Las fechas fueron convertidas a formato datetime para permitir agregaciones y gráficos correctos.


In [ ]:
users_in_both = (orders.groupby('visitorId')['group']
                 .nunique()
                 .reset_index()
                 .query('group > 1')['visitorId'])

print("Usuarios en ambos grupos:", users_in_both.shape[0])

orders_clean = orders[~orders['visitorId'].isin(users_in_both)]


Se identificaron visitantes que aparecen en ambos grupos (A y B). Para evitar contaminación del experimento, se eliminaron sus pedidos del análisis.

## Parte 1. Priorizar hipótesis

### Aplicar el framework ICE para priorizar hipótesis. Ordenarlas en orden descendente de prioridad.

In [ ]:
hyp['ICE'] = (hyp['Impact'] * hyp['Confidence']) / hyp['Effort']
ice_rank = hyp[['Hypothesis','Reach','Impact','Confidence','Effort','ICE']].sort_values('ICE', ascending=False)
ice_rank


Se aplicó el framework ICE para priorizar las hipótesis considerando su impacto potencial, el nivel de confianza y el esfuerzo requerido para implementarlas.

Las hipótesis con mayor puntuación ICE presentan una combinación de alto impacto y confianza con un esfuerzo relativamente bajo. En particular, la hipótesis relacionada con ofrecer descuentos en cumpleaños obtuvo la mayor prioridad, ya que combina un impacto y confianza elevados con un esfuerzo moderado.

Por el contrario, las hipótesis con bajo impacto o alto esfuerzo quedaron relegadas a posiciones inferiores del ranking.

### Aplicar el framework RICE para priorizar hipótesis. Ordenarlas en orden descendente de prioridad.

In [ ]:
hyp['RICE'] = (hyp['Reach'] * hyp['Impact'] * hyp['Confidence']) / hyp['Effort']
rice_rank = hyp[['Hypothesis','Reach','Impact','Confidence','Effort','RICE']].sort_values('RICE', ascending=False)
rice_rank


Se aplicó el framework RICE para priorizar las hipótesis considerando el alcance (Reach), el impacto potencial en los usuarios (Impact), el nivel de confianza (Confidence) y el esfuerzo requerido para su implementación (Effort).

Este enfoque permite identificar las iniciativas que no solo tienen un impacto positivo, sino que además afectan a un mayor número de usuarios, lo cual resulta especialmente relevante para maximizar el crecimiento del negocio.

Los resultados muestran que la hipótesis de agregar un formulario de suscripción en todas las páginas principales obtuvo la puntuación RICE más alta, debido a su alto alcance combinado con un impacto significativo y un esfuerzo moderado. Otras hipótesis relacionadas con recomendaciones de productos y nuevos canales de atracción de tráfico también obtuvieron puntuaciones elevadas, al beneficiar a una gran parte de los usuarios.

En contraste, las hipótesis con bajo alcance, como la promoción de descuentos en cumpleaños, descendieron en el ranking a pesar de su alto impacto individual, lo que resalta la importancia del factor Reach en la priorización.

### Mostrar cómo cambia la priorización de hipótesis cuando utilizas RICE en lugar de ICE. Proporcionar una explicación de los cambios.

Al comparar los frameworks ICE y RICE, se observa un cambio significativo en la priorización de las hipótesis.

ICE se enfoca únicamente en impacto, confianza y esfuerzo, lo que favorece hipótesis con alto impacto aunque afecten a un número reducido de usuarios. Por ejemplo, la hipótesis de ofrecer descuentos en cumpleaños obtuvo la puntuación ICE más alta, pero cayó varias posiciones al aplicar RICE debido a su bajo alcance.

En contraste, RICE incorpora el factor Reach, priorizando hipótesis que impactan a un mayor número de usuarios. Como resultado, la hipótesis de agregar un formulario de suscripción en todas las páginas principales pasó a ocupar el primer lugar, ya que combina un alto alcance con buen impacto y esfuerzo moderado.

Esto demuestra que RICE proporciona una priorización más orientada al impacto global del negocio.

## Parte 2. Análisis del test A/B

Realizaste una prueba A/B y obtuviste los resultados descritos en los archivos orders_us.csv y visits_us.csv.

In [ ]:
users_in_both = (
    orders.groupby('visitorId')['group']
    .nunique()
    .reset_index()
    .query('group > 1')['visitorId']
)

print('Usuarios en ambos grupos:', users_in_both.shape[0])

orders_clean = orders[~orders['visitorId'].isin(users_in_both)]


Antes de analizar los resultados del test A/B, se identificaron visitantes que participaron en ambos grupos (A y B). Estos usuarios fueron eliminados del análisis para evitar sesgos y garantizar la validez del experimento.

### 1. Representa gráficamente el ingreso acumulado por grupo. Haz conclusiones y conjeturas.

In [ ]:
daily_revenue = (
    orders_clean
    .groupby(['date', 'group'], as_index=False)
    .agg({'revenue': 'sum'})
)

daily_revenue['cumulative_revenue'] = (
    daily_revenue
    .groupby('group')['revenue']
    .cumsum()
)

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
for group in ['A', 'B']:
    data = daily_revenue[daily_revenue['group'] == group]
    plt.plot(data['date'], data['cumulative_revenue'], label=f'Group {group}')

plt.title('Ingreso acumulado por grupo')
plt.xlabel('Fecha')
plt.ylabel('Ingreso acumulado')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


El gráfico de ingreso acumulado muestra que ambos grupos presentan una tendencia creciente a lo largo del periodo del experimento. Durante los primeros días, los ingresos acumulados de los grupos A y B se mantienen relativamente cercanos.

Sin embargo, a partir de mediados del periodo, el grupo B comienza a separarse claramente del grupo A, mostrando un ingreso acumulado significativamente mayor. Este comportamiento se ve reforzado por un incremento abrupto en el ingreso del grupo B alrededor del 18 de agosto, lo que sugiere la presencia de uno o varios pedidos de alto valor.

A pesar de este salto puntual, el grupo B mantiene una ventaja consistente sobre el grupo A hasta el final del experimento. Esto indica que el grupo B genera mayores ingresos acumulados, aunque parte de la diferencia podría estar influenciada por pedidos atípicos, los cuales deberán analizarse posteriormente mediante la detección de anomalías.



### 2. Representa gráficamente el tamaño de pedido promedio acumulado por grupo. Haz conclusiones y conjeturas.

In [ ]:
daily_orders = (
    orders_clean
    .groupby(['date', 'group'], as_index=False)
    .agg({'transactionId': 'nunique', 'revenue': 'sum'})
)

daily_orders['cumulative_revenue'] = (
    daily_orders
    .groupby('group')['revenue']
    .cumsum()
)

daily_orders['cumulative_orders'] = (
    daily_orders
    .groupby('group')['transactionId']
    .cumsum()
)

daily_orders['cumulative_aov'] = (
    daily_orders['cumulative_revenue'] /
    daily_orders['cumulative_orders']
)

plt.figure(figsize=(10, 5))
for group in ['A', 'B']:
    data = daily_orders[daily_orders['group'] == group]
    plt.plot(data['date'], data['cumulative_aov'], label=f'Group {group}')

plt.title('Tamaño de pedido promedio acumulado')
plt.xlabel('Fecha')
plt.ylabel('AOV acumulado')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


El gráfico del tamaño de pedido promedio (AOV) acumulado muestra que ambos grupos presentan una evolución relativamente estable durante la primera mitad del experimento, con valores similares y algunas fluctuaciones.

Sin embargo, a partir de aproximadamente el 18 de agosto, el grupo B presenta un aumento abrupto en el AOV acumulado, superando de forma significativa al grupo A. Este cambio repentino sugiere la presencia de uno o varios pedidos de alto valor que influyen considerablemente en el promedio acumulado del grupo B.

Después de este incremento, el AOV del grupo B comienza a descender gradualmente, aunque se mantiene por encima del grupo A hasta el final del periodo. En contraste, el grupo A muestra un comportamiento más estable, sin variaciones extremas.

En conjunto, estos resultados indican que el AOV del grupo B es mayor, pero que esta diferencia podría estar influenciada por valores atípicos, por lo que es necesario analizar y filtrar anomalías antes de sacar conclusiones definitivas.



### 3. Representa gráficamente la diferencia relativa en el tamaño de pedido promedio acumulado para el grupo B en comparación con el grupo A. Haz conclusiones y conjeturas.

In [ ]:
pivot_aov = daily_orders.pivot(
    index='date',
    columns='group',
    values='cumulative_aov'
)

pivot_aov['relative_difference'] = (
    pivot_aov['B'] / pivot_aov['A'] - 1
)

plt.figure(figsize=(10, 5))
plt.plot(pivot_aov.index, pivot_aov['relative_difference'])
plt.axhline(0, linestyle='--')
plt.title('Diferencia relativa del AOV acumulado (B vs A)')
plt.xlabel('Fecha')
plt.ylabel('Diferencia relativa')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


El gráfico de la diferencia relativa del tamaño de pedido promedio (AOV) acumulado entre los grupos B y A muestra un comportamiento altamente volátil durante el periodo del experimento.

Al inicio, la diferencia fluctúa alrededor de cero, alternando entre valores positivos y negativos, lo que indica que ninguno de los grupos presenta una ventaja clara en el AOV acumulado durante los primeros días.

A partir de aproximadamente el 18 de agosto, se observa un incremento abrupto en la diferencia relativa, alcanzando valores cercanos al 50%. Este cambio coincide con la aparición de pedidos de alto valor en el grupo B, lo que provoca una sobreestimación temporal del AOV acumulado en comparación con el grupo A.

Posteriormente, la diferencia relativa comienza a disminuir gradualmente, aunque se mantiene en valores positivos hasta el final del periodo. Esto sugiere que el AOV del grupo B es mayor que el del grupo A, pero que dicha diferencia está fuertemente influenciada por valores atípicos. Por lo tanto, es necesario analizar los datos filtrados antes de extraer conclusiones definitivas.



### 4. Calcula la tasa de conversión de cada grupo como la relación entre los pedidos y el número de visitas de cada día. Representa gráficamente las tasas de conversión diarias de los dos grupos y describe la diferencia. Saca conclusiones y haz conjeturas.

In [ ]:
# pedidos diarios por grupo
daily_orders = (
    orders_clean
    .groupby(['date', 'group'], as_index=False)
    .agg({'transactionId': 'nunique'})
)

# unir con visitas
daily_conversion = visits.merge(
    daily_orders,
    on=['date', 'group'],
    how='left'
)

# reemplazar NaN por 0 pedidos
daily_conversion['transactionId'] = daily_conversion['transactionId'].fillna(0)

# calcular conversión
daily_conversion['conversion'] = (
    daily_conversion['transactionId'] / daily_conversion['visits']
)

daily_conversion.head()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

for group in ['A', 'B']:
    data = daily_conversion[daily_conversion['group'] == group]
    plt.plot(data['date'], data['conversion'], label=f'Group {group}')

plt.title('Tasa de conversión diaria por grupo')
plt.xlabel('Fecha')
plt.ylabel('Conversión')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


La tasa de conversión diaria se calculó como la relación entre el número de pedidos y el número de visitas para cada grupo. El gráfico muestra que la conversión presenta fluctuaciones diarias en ambos grupos, lo cual es esperable debido a la variabilidad natural del tráfico y las compras.

A lo largo del periodo del experimento, el grupo B muestra una tasa de conversión generalmente superior a la del grupo A en la mayoría de los días, aunque la diferencia no es completamente estable y existen días en los que ambos grupos presentan valores similares.

No se observa una separación extremadamente marcada entre las curvas, lo que sugiere que, aunque el grupo B podría tener una ligera ventaja en conversión, esta diferencia debe ser evaluada mediante pruebas de significancia estadística.

En conjunto, los resultados preliminares indican que el grupo B podría estar convirtiendo mejor que el grupo A, pero es necesario confirmar esta hipótesis utilizando métodos estadísticos y considerando el efecto de posibles valores atípicos.



### 5. Traza un gráfico de dispersión del número de pedidos por usuario. Haz conclusiones y conjeturas.

In [ ]:
orders_by_user = (
    orders_clean
    .groupby('visitorId', as_index=False)
    .agg({'transactionId': 'nunique'})
)

orders_by_user.columns = ['visitorId', 'orders']
orders_by_user.head()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.scatter(range(len(orders_by_user)), orders_by_user['orders'])
plt.title('Número de pedidos por usuario')
plt.xlabel('Usuarios (índice)')
plt.ylabel('Número de pedidos')
plt.tight_layout()
plt.show()


El gráfico de dispersión del número de pedidos por usuario muestra que la gran mayoría de los usuarios realizó un solo pedido durante el periodo del experimento.

Un número reducido de usuarios realizó dos pedidos, y solo unos pocos usuarios realizaron tres pedidos, lo que los coloca claramente separados del resto de la distribución.

Estos usuarios con un número inusualmente alto de pedidos pueden considerarse valores atípicos, ya que su comportamiento no es representativo del patrón general de compra. Su presencia puede influir de manera desproporcionada en métricas como los ingresos acumulados y la tasa de conversión.

Por esta razón, resulta necesario definir umbrales objetivos utilizando percentiles para determinar qué usuarios deben ser considerados anomalías y evaluar los resultados del test A/B tanto con datos en bruto como con datos filtrados.



### 6. Calcula los percentiles 95 y 99 del número de pedidos por usuario. Define el punto en el cual un punto de datos se convierte en una anomalía.

In [ ]:
# percentiles 95 y 99
np.percentile(orders_by_user['orders'], [95, 99])


Se calcularon los percentiles 95 y 99 del número de pedidos por usuario con el objetivo de identificar comportamientos atípicos. El percentil 95 corresponde a un pedido, mientras que el percentil 99 corresponde a dos pedidos.

Esto indica que el 99% de los usuarios realizó dos pedidos o menos durante el periodo del experimento. En consecuencia, se definió como anomalía a cualquier usuario que haya realizado más de dos pedidos, ya que este comportamiento no es representativo del conjunto general de usuarios.

Este criterio permite reducir la influencia de valores extremos en las métricas del test A/B y realizar un análisis más robusto utilizando datos filtrados.



### 7. Traza un gráfico de dispersión de los precios de los pedidos. Haz conclusiones y conjeturas.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.scatter(range(len(orders_clean)), orders_clean['revenue'])
plt.title('Precios de los pedidos')
plt.xlabel('Pedidos (índice)')
plt.ylabel('Revenue')
plt.tight_layout()
plt.show()


El gráfico de dispersión de los precios de los pedidos muestra que la gran mayoría de los pedidos se concentran en valores relativamente bajos de revenue, formando una distribución claramente sesgada hacia la derecha.

Se observa una cola larga con un pequeño número de pedidos de precio significativamente más alto que el resto, incluyendo uno extremadamente alto que se encuentra muy por encima del conjunto principal de datos. Estos pedidos representan valores atípicos que pueden influir de manera desproporcionada en métricas como el ingreso total y el tamaño de pedido promedio.

La presencia de estos pedidos de alto valor refuerza la necesidad de definir umbrales objetivos mediante percentiles para identificar anomalías y evaluar los resultados del test A/B tanto con datos en bruto como con datos filtrados.



### 8. Calcula los percentiles 95 y 99 de los precios de los pedidos. Define el punto en el cual un punto de datos se convierte en una anomalía.

In [ ]:
np.percentile(orders_clean['revenue'], [95, 99])


Se calcularon los percentiles 95 y 99 de los precios de los pedidos para identificar valores atípicos. El percentil 95 corresponde a un precio de 414.28, mientras que el percentil 99 corresponde a un precio de 830.30.

Estos resultados indican que el 99% de los pedidos tiene un precio menor o igual a 830.30, y que solo un pequeño porcentaje de pedidos presenta valores significativamente más altos.

Con base en este análisis, se definió como anomalía a cualquier pedido cuyo precio sea mayor que 830.30, ya que estos valores pueden influir de manera desproporcionada en métricas como el ingreso total y el tamaño de pedido promedio.

Este umbral permite realizar un análisis más robusto del test A/B al evaluar los resultados tanto con datos en bruto como con datos filtrados.



### 9. Encuentra la significancia estadística de la diferencia en la conversión entre los grupos utilizando los datos en bruto. Haz conclusiones y conjeturas.

In [ ]:
from scipy import stats
import pandas as pd

# pedidos por usuario y grupo
orders_by_user = (
    orders_clean
    .groupby(['group', 'visitorId'], as_index=False)
    .agg({'transactionId': 'nunique'})
)

orders_by_user.columns = ['group', 'visitorId', 'orders']

# muestras de pedidos
users_a = orders_by_user[orders_by_user['group'] == 'A']['orders']
users_b = orders_by_user[orders_by_user['group'] == 'B']['orders']

# número total de visitas por grupo (proxy de usuarios)
visits_a = visits[visits['group'] == 'A']['visits'].sum()
visits_b = visits[visits['group'] == 'B']['visits'].sum()

# agregar ceros para usuarios sin pedidos
sample_a = pd.concat(
    [users_a, pd.Series(0, index=range(int(visits_a - users_a.shape[0])))],
    ignore_index=True
)

sample_b = pd.concat(
    [users_b, pd.Series(0, index=range(int(visits_b - users_b.shape[0])))],
    ignore_index=True
)


In [ ]:
alpha = 0.05

p_value = stats.mannwhitneyu(sample_a, sample_b, alternative='two-sided')[1]

conversion_a = sample_a.mean()
conversion_b = sample_b.mean()

relative_lift = conversion_b / conversion_a - 1

p_value, relative_lift


Para evaluar la significancia estadística de la diferencia en la tasa de conversión entre los grupos A y B, se utilizó la prueba no paramétrica de Mann–Whitney U, dado que la distribución de los datos no es normal y contiene una gran cantidad de valores cero.

El resultado de la prueba arrojó un valor p de 0.011, el cual es menor que el nivel de significancia de 0.05. Esto indica que la diferencia observada en la tasa de conversión entre los grupos es estadísticamente significativa al utilizar los datos en bruto.

Adicionalmente, el cambio relativo en la conversión del grupo B en comparación con el grupo A fue de aproximadamente 16%, lo que sugiere que el grupo B presenta una tasa de conversión superior.

En conjunto, estos resultados indican que existe evidencia estadística suficiente para afirmar que el grupo B convierte mejor que el grupo A cuando se consideran los datos sin filtrar.



### 10. Encuentra la significancia estadística de la diferencia en el tamaño promedio de pedido entre los grupos utilizando los datos en bruto. Haz conclusiones y conjeturas.

In [ ]:
from scipy import stats

# revenue por pedido por grupo
revenue_a = orders_clean[orders_clean['group'] == 'A']['revenue']
revenue_b = orders_clean[orders_clean['group'] == 'B']['revenue']


In [ ]:

alpha = 0.05

p_value_aov = stats.mannwhitneyu(
    revenue_a,
    revenue_b,
    alternative='two-sided'
)[1]

mean_aov_a = revenue_a.mean()
mean_aov_b = revenue_b.mean()

relative_diff_aov = mean_aov_b / mean_aov_a - 1

p_value_aov, relative_diff_aov


Para evaluar la significancia estadística de la diferencia en el tamaño promedio de pedido entre los grupos A y B, se utilizó la prueba no paramétrica de Mann–Whitney U, dado que la distribución de los precios de los pedidos es asimétrica y presenta valores atípicos.

El resultado de la prueba arrojó un valor p de 0.862, el cual es considerablemente mayor que el nivel de significancia de 0.05. Esto indica que la diferencia observada en el tamaño promedio de pedido entre los grupos no es estadísticamente significativa cuando se utilizan los datos en bruto.

Si bien el tamaño promedio de pedido del grupo B es aproximadamente 27.8% mayor que el del grupo A, esta diferencia no puede considerarse estadísticamente confiable y probablemente está influenciada por pedidos de alto valor.

En consecuencia, no existe evidencia estadística suficiente para afirmar que el grupo B tenga un tamaño promedio de pedido mayor que el grupo A al analizar los datos sin filtrar.



### 11. Encuentra la significancia estadística de la diferencia en la conversión entre los grupos utilizando los datos filtrados. Haz conclusiones y conjeturas.

Datos importantes:

- Usuarios anómalos: más de 2 pedidos

- Pedidos anómalos: revenue > 830.3

In [ ]:
# usuarios con demasiados pedidos
orders_per_user = (
    orders_clean
    .groupby('visitorId', as_index=False)
    .agg({'transactionId': 'nunique'})
)
orders_per_user.columns = ['visitorId', 'orders']

abnormal_users = orders_per_user[orders_per_user['orders'] > 2]['visitorId']

# pedidos con revenue anómalo
abnormal_orders = orders_clean[orders_clean['revenue'] > 830.3]['visitorId']

# unir anomalías
abnormal_visitors = pd.concat(
    [abnormal_users, abnormal_orders]
).drop_duplicates()

# datos filtrados
orders_filtered = orders_clean[
    ~orders_clean['visitorId'].isin(abnormal_visitors)
]


In [ ]:
# pedidos por usuario filtrados
orders_users_f = (
    orders_filtered
    .groupby(['group', 'visitorId'], as_index=False)
    .agg({'transactionId': 'nunique'})
)

orders_users_f.columns = ['group', 'visitorId', 'orders']

users_a_f = orders_users_f[orders_users_f['group'] == 'A']['orders']
users_b_f = orders_users_f[orders_users_f['group'] == 'B']['orders']

# visitas totales (proxy usuarios)
visits_a = visits[visits['group'] == 'A']['visits'].sum()
visits_b = visits[visits['group'] == 'B']['visits'].sum()

# agregar ceros (usuarios sin pedidos)
sample_a_f = pd.concat(
    [users_a_f, pd.Series(0, index=range(int(visits_a - users_a_f.shape[0])))],
    ignore_index=True
)

sample_b_f = pd.concat(
    [users_b_f, pd.Series(0, index=range(int(visits_b - users_b_f.shape[0])))],
    ignore_index=True
)


In [ ]:
from scipy import stats

p_value_conv_f = stats.mannwhitneyu(
    sample_a_f,
    sample_b_f,
    alternative='two-sided'
)[1]

conversion_a_f = sample_a_f.mean()
conversion_b_f = sample_b_f.mean()

relative_lift_f = conversion_b_f / conversion_a_f - 1

p_value_conv_f, relative_lift_f


Para evaluar la significancia estadística de la diferencia en la tasa de conversión entre los grupos A y B utilizando datos filtrados, se aplicó la prueba no paramétrica de Mann–Whitney U. Previamente, se excluyeron usuarios con comportamientos atípicos, definidos como aquellos con más de dos pedidos o con pedidos cuyo precio supera el percentil 99.

El resultado de la prueba arrojó un valor p de 0.00695, el cual es menor que el nivel de significancia de 0.05. Esto indica que la diferencia observada en la tasa de conversión entre los grupos es estadísticamente significativa incluso después de eliminar los valores atípicos.

Asimismo, el cambio relativo en la conversión del grupo B en comparación con el grupo A fue de aproximadamente 18.9%, lo que sugiere que el grupo B mantiene una ventaja clara en términos de conversión.

En conjunto, estos resultados indican que la mejora en la conversión del grupo B es robusta y no está influenciada por comportamientos atípicos.



### 12. Encuentra la significancia estadística de la diferencia en el tamaño promedio de pedido entre los grupos utilizando los datos filtrados. Haz conclusiones y conjeturas.

In [ ]:
from scipy import stats

# revenue por pedido usando datos filtrados
revenue_a_f = orders_filtered[orders_filtered['group'] == 'A']['revenue']
revenue_b_f = orders_filtered[orders_filtered['group'] == 'B']['revenue']


In [ ]:
p_value_aov_f = stats.mannwhitneyu(
    revenue_a_f,
    revenue_b_f,
    alternative='two-sided'
)[1]

mean_aov_a_f = revenue_a_f.mean()
mean_aov_b_f = revenue_b_f.mean()

relative_diff_aov_f = mean_aov_b_f / mean_aov_a_f - 1

p_value_aov_f, relative_diff_aov_f


Para evaluar la significancia estadística de la diferencia en el tamaño promedio de pedido entre los grupos A y B utilizando datos filtrados, se aplicó la prueba no paramétrica de Mann–Whitney U, tras excluir pedidos y usuarios considerados atípicos.

El resultado de la prueba arrojó un valor p de 0.822, el cual es mayor que el nivel de significancia de 0.05. Esto indica que la diferencia observada en el tamaño promedio de pedido entre los grupos no es estadísticamente significativa al utilizar los datos filtrados.

La diferencia relativa en el tamaño promedio de pedido del grupo B en comparación con el grupo A fue de aproximadamente −3.2%, lo que sugiere que el AOV del grupo B es ligeramente menor; sin embargo, esta diferencia es pequeña y no puede considerarse estadísticamente relevante.

En conjunto, estos resultados indican que, tras eliminar los valores atípicos, no se observa una diferencia significativa en el tamaño promedio de pedido, lo que confirma que el principal impacto del experimento se encuentra en la tasa de conversión y no en el valor de los pedidos individuales.



### 13. Toma una decisión basada en los resultados de la prueba. Las decisiones posibles son: 1. Parar la prueba, considerar a uno de los grupos como líder. 2. Parar la prueba, concluir que no hay diferencia entre los grupos. 3. Continuar la prueba.

**Decisión**

Con base en los resultados del test A/B, se decide parar la prueba y considerar al grupo B como líder.

**Justificación**

La tasa de conversión del grupo B es significativamente mayor que la del grupo A tanto en los datos en bruto como en los datos filtrados, lo que indica que el efecto observado es robusto y no está influenciado por valores atípicos.

En contraste, no se encontraron diferencias estadísticamente significativas en el tamaño promedio de pedido entre los grupos, lo que sugiere que el impacto del experimento se concentra en mejorar la conversión y no en aumentar el valor de los pedidos individuales.

Dado que la conversión es una métrica clave para el crecimiento del negocio y que el grupo B muestra una mejora consistente y estadísticamente significativa, se concluye que el grupo B debe considerarse el ganador del experimento.